[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/warlockee/oxRL/blob/main/examples/oxrl_quickstart.ipynb)

# oxRL Quickstart: Post-train any LLM in minutes

This notebook demonstrates how to post-train a language model using **oxRL** -- a lightweight framework with 51 algorithms for RL and preference optimization.

**What you will do:**
1. Run SFT (Supervised Fine-Tuning) on a small model
2. See how to configure DPO and other preference algorithms
3. Browse the full 51-algorithm catalog

**Requirements:** A single GPU (T4 or better). Runs in under 5 minutes on Colab.

> **Note:** The RL path (GRPO/PPO) requires Ray + vLLM and is designed for multi-GPU servers. This notebook demonstrates the SL path which works on a single Colab GPU.

---
## 1. Install oxRL

In [ ]:
!pip install oxrl deepspeed -q

---
## 2. SFT Training

Let's fine-tune **Qwen2.5-0.5B-Instruct** on a small instruction dataset. We'll write a config and launch training with DeepSpeed.

First, let's prepare a tiny demo dataset:

In [ ]:
import json, os

# Create a small demo dataset (10 examples)
os.makedirs("/tmp/oxrl_demo", exist_ok=True)

examples = [
    {"prompt": "What is 2+2?", "answer": "4"},
    {"prompt": "What is the capital of France?", "answer": "Paris"},
    {"prompt": "Translate 'hello' to Spanish.", "answer": "hola"},
    {"prompt": "What is 3*7?", "answer": "21"},
    {"prompt": "Name a primary color.", "answer": "Red"},
    {"prompt": "What is H2O?", "answer": "Water"},
    {"prompt": "What planet is closest to the Sun?", "answer": "Mercury"},
    {"prompt": "What is 10-4?", "answer": "6"},
    {"prompt": "What language is spoken in Brazil?", "answer": "Portuguese"},
    {"prompt": "What is the square root of 9?", "answer": "3"},
]

for split in ["train", "val"]:
    with open(f"/tmp/oxrl_demo/{split}.jsonl", "w") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")

print(f"Created demo dataset: {len(examples)} examples")

In [ ]:
%%writefile /tmp/oxrl_demo/sft_config.yaml
run:
  experiment_id: quickstart_sft
  distributed_training_strategy: deepspeed-zero2
  checkpoint_dir: /tmp/oxrl_demo/checkpoints
  project_name: oxrl-quickstart
  tracking_uri: ""
  seed: 42

train:
  alg_name: sft
  lr: 2.0e-05
  total_number_of_epochs: 1
  micro_batches_per_epoch: 10
  train_batch_size_per_gpu: 2
  gradient_accumulation_steps: 1
  val_batch_size_per_gpu: 4

model:
  name: Qwen/Qwen2.5-0.5B-Instruct
  dtype: bfloat16
  trust_remote_code: true
  use_cache: false
  model_class: llm
  gradient_checkpointing: false
  attn_implementation: eager

data:
  train_dnames: [demo]
  train_files_path: /tmp/oxrl_demo/train.jsonl
  val_files_path: /tmp/oxrl_demo/val.jsonl
  num_workers: 0
  max_seq_len: 256
  prompt_key: prompt
  answer_key: answer

deepspeed:
  zero_optimization:
    stage: 2
    offload_optimizer:
      device: none
    contiguous_gradients: true
    overlap_comm: true
    reduce_scatter: true
    reduce_bucket_size: 500000000.0
    allgather_bucket_size: 500000000.0
  steps_per_print: 100
  wall_clock_breakdown: false

In [ ]:
%%time
# Launch SFT training with DeepSpeed
!python -m oxrl.main_sl --config-file /tmp/oxrl_demo/sft_config.yaml --experiment_id quickstart_sft

---
## 3. Switching Algorithms

To use a different algorithm, just change `alg_name` in the config. For preference algorithms like DPO, your dataset needs `chosen` and `rejected` columns.

Examples:
- `alg_name: dpo` -- Direct Preference Optimization (needs ref model + preference pairs)
- `alg_name: simpo` -- Reference-free, no ref model needed
- `alg_name: orpo` -- Odds ratio preference, reference-free
- `alg_name: kto` -- Works with binary good/bad labels instead of pairs
- `alg_name: cpo` -- Contrastive PO, reference-free with NLL regularizer

All 45 SL algorithms use the same config structure. See `registry/examples/` for ready-made configs.

For **RL training** (GRPO, PPO) on a multi-GPU server:
```python
from oxrl import Trainer
trainer = Trainer(model="deepseek-ai/DeepSeek-R1-Distill-Llama-8B")
trainer.train(task="reasoning")
```

---
## 4. Algorithm Catalog (51 algorithms)

oxRL ships with **51 algorithms** across two training paradigms.

### RL Algorithms (online, policy-gradient via `main_rl.py`)

These generate rollouts online using vLLM, score them with a reward function, and update the policy.

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 1 | **SGRPO** | `sgrpo` | Token-level clipped surrogate (default for dense models) |
| 2 | **GSPO** | `gspo` | Sequence-level clipped surrogate (for MoE models) |
| 3 | **CISPO** | `cispo` | Clipped ratio weighted log-prob (conservative variant) |
| 4 | **RLHF** | `rlhf` | RL from Human Feedback (alias for SGRPO + reward model) |
| 5 | **RLAIF** | `rlaif` | RL from AI Feedback (alias for SGRPO) |
| 6 | **PPO** | `ppo` | Proximal Policy Optimization with value head |

### SL Algorithms (offline, preference/supervised via `main_sl.py`)

These train on static datasets without rollout generation.

#### Core Training

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 7 | **SFT** | `sft` | Supervised fine-tuning |
| 8 | **CPT** | `cpt` | Continued pre-training |
| 9 | **KD** | `kd` | Knowledge distillation |
| 10 | **RFT** | `rft` | Rejection sampling fine-tuning |
| 11 | **RM** | `rm` | Reward model training (Bradley-Terry) |

#### DPO and Direct Variants

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 12 | **DPO** | `dpo` | Direct Preference Optimization |
| 13 | **IPO** | `ipo` | Identity Preference Optimization |
| 14 | **RDPO** | `rdpo` | Length-regularized DPO |
| 15 | **cDPO** | `cdpo` | Conservative DPO (label smoothing) |
| 16 | **ODPO** | `odpo` | DPO with offset |
| 17 | **DPOP** | `dpop` | DPO-Positive / Smaug |
| 18 | **DPO-Shift** | `dposhift` | Distribution-shifted DPO |
| 19 | **DPO+NLL** | `dpnll` | DPO with NLL regularization |
| 20 | **C2-DPO** | `c2dpo` | Constrained Controlled DPO |

#### Robust and Calibrated Variants

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 21 | **BetaDPO** | `betadpo` | Dynamic beta per sample |
| 22 | **CalDPO** | `caldpo` | Calibrated DPO |
| 23 | **RobustDPO** | `robust_dpo` | Unbiased DPO under label noise |
| 24 | **DrDPO** | `drdpo` | Distributionally Robust DPO (ICLR 2025) |
| 25 | **AlphaDPO** | `alpha_dpo` | Adaptive reward margin (ICML 2025) |
| 26 | **MinorDPO** | `minor_dpo` | Minority-aware DPO |

#### Reference-Free Methods

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 27 | **SimPO** | `simpo` | Simple Preference Optimization (no ref model) |
| 28 | **ORPO** | `orpo` | Odds Ratio Preference Optimization |
| 29 | **CPO** | `cpo` | Contrastive Preference Optimization |
| 30 | **CPO-SimPO** | `cposimpo` | CPO + SimPO combined |
| 31 | **AlphaPO** | `alphapo` | Reward shape transformation for SimPO |

#### Alternative Loss Functions

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 32 | **Hinge** | `hinge` | Hinge loss preference optimization |
| 33 | **NCA** | `nca` | Noise Contrastive Alignment |
| 34 | **SPPO** | `sppo` | Self-Play Preference Optimization |
| 35 | **BCO** | `bco` | Binary Classifier Optimization |
| 36 | **EXO** | `exo` | Reverse KL preference optimization |
| 37 | **AOT** | `aot` | Alignment via Optimal Transport |
| 38 | **APO** | `apo` | Anchored Preference Optimization |

#### Generalized and Focal Methods

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 39 | **GPO** | `gpo` | Generalized Preference Optimization (ICML 2024) |
| 40 | **FocalPO** | `focalpo` | Focal preference optimization |
| 41 | **WPO** | `wpo` | Weighted Preference Optimization (EMNLP 2024) |
| 42 | **fDPO** | `fdpo` | f-Divergence DPO (ICLR 2024) |
| 43 | **HDPO** | `hdpo` | Entropy Controllable DPO |
| 44 | **DiscoPOP** | `discopop` | Log-ratio modulated loss |
| 45 | **BPO** | `bpo` | Balanced Preference Optimization |
| 46 | **SamPO** | `sampo` | Sample-level Preference Optimization |
| 47 | **ChiPO** | `chipo` | Chi-squared Preference Optimization |
| 48 | **SPO** | `spo` | Spectral Preference Optimization |

#### Other Training Methods

| # | Algorithm | Key | Description |
|---|-----------|-----|-------------|
| 49 | **KTO** | `kto` | Kahneman-Tversky Optimization |
| 50 | **OnlineDPO** | `online_dpo` | Online DPO with iterative data |
| 51 | **SPIN** | `spin` | Self-Play Fine-Tuning |


---
## 5. Links

[![GitHub stars](https://img.shields.io/github/stars/warlockee/oxRL?style=social)](https://github.com/warlockee/oxRL)

- **GitHub:** [github.com/warlockee/oxRL](https://github.com/warlockee/oxRL)
- **PyPI:** [pypi.org/project/oxrl](https://pypi.org/project/oxrl/)
- **Docs:** See `registry/examples/` for ready-made configs for every algorithm